# Inspect highway sensing

Compare the same fixed drive in clear, rain, and fog. The simulator view and the policy's occupancy map appear side by side.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
here = Path.cwd().resolve()
repo = next((p for p in (here, *here.parents) if (p / "runs/index.csv").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the MORL4AVSO checkout.")
sys.path.insert(0, str(repo))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from environments.sensor_highway import ExperimentConfig
from experiments.notebook_helpers import plot_snapshot, run_rollout, summarize

In [ ]:
SEED = 42
WEATHER = ("clear", "rain", "fog")
SENSING_MODES = {"all on": "all_on", "front": "front", "standby": "off"}
config = ExperimentConfig()
config.highway["duration"] = 8

rows = []
for weather in WEATHER:
    config.weather.preset = weather
    for label, mode in SENSING_MODES.items():
        rollout = run_rollout(config, seed=SEED, mode=mode)
        rows.append({"weather": weather, "sensing": label, **summarize(rollout)})

metrics = ["mean_power_w", "mean_supported_fraction", "mean_uncertainty"]
display(pd.DataFrame(rows).set_index(["weather", "sensing"])[metrics].round(3))

In [ ]:
config.weather.preset = "fog"
rollout = run_rollout(config, seed=SEED, mode="front", capture=True)
plot_snapshot(rollout)
plt.show()